# 05 – Generate Gold Labels
This notebook generates/manages gold labels for evaluation (questions, expected answers, and reference Cypher).


In [1]:
import json
import os
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
from neo4j import GraphDatabase
from tqdm import tqdm

# -----------------------------
# Config
# -----------------------------
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATASETS_DIR = ROOT / 'tests' / 'datasets'
INPUT_PATH = DATASETS_DIR / 'eval_dataset.jsonl'
OUTPUT_PATH = DATASETS_DIR / 'eval_dataset_with_gold.jsonl'

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

# LIST output is intentionally capped (mirrors your generator rules)
LIST_MAX_LINES = 20


In [2]:
# -----------------------------
# IO helpers: JSONL or JSON-array
# -----------------------------
def load_dataset(path: Path) -> List[Dict[str, Any]]:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    # JSON array
    if text.startswith('['):
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON array expected")
        return data

    # JSONL
    rows: List[Dict[str, Any]] = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows


def save_json_array(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def save_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

In [3]:
# -----------------------------
# Load dataset
# -----------------------------
dataset = load_dataset(INPUT_PATH)
len(dataset), dataset[0].keys() if dataset else None

(100, dict_keys(['id', 'category', 'question']))

In [4]:
# -----------------------------
# Neo4j helpers
# -----------------------------
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(cypher: str, params: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    params = params or {}
    with driver.session() as session:
        return session.run(cypher, params).data()

# quick connectivity check (prints server version)
run_cypher('RETURN 1 AS ok')[0]

{'ok': 1}

In [5]:
# -----------------------------
# Parsing helpers (from English questions)
# -----------------------------
_RE_EPOCH = re.compile(r"epoch\s*(\d+)", re.IGNORECASE)
_RE_SENSOR = re.compile(r"sensor\s*(\d+(?:\.0)?)", re.IGNORECASE)
_RE_DATE = re.compile(r"(\d{4}-\d{2}-\d{2})")
_RE_TIME = re.compile(r"(\d{2}:\d{2}:\d{2})")

def parse_epoch(q: str) -> Optional[int]:
    m = _RE_EPOCH.search(q or '')
    return int(m.group(1)) if m else None

def parse_sensor(q: str) -> Optional[int]:
    m = _RE_SENSOR.search(q or '')
    if not m:
        return None
    raw = m.group(1)
    try:
        return int(float(raw))
    except Exception:
        return None

def parse_datetime_prefix(q: str) -> Optional[str]:
    d = _RE_DATE.search(q or '')
    t = _RE_TIME.search(q or '')
    if not (d and t):
        return None
    # Measurement.time stored as 'YYYY-MM-DDTHH:MM:SS...' (may include fractions)
    return f"{d.group(1)}T{t.group(1)}"

def parse_metric(q: str) -> Optional[str]:
    ql = (q or '').lower()
    if 'temperature' in ql:
        return 'temperature'
    if 'humidity' in ql:
        return 'humidity'
    if 'light' in ql:
        return 'light'
    if 'voltage' in ql:
        return 'voltage'
    return None

def detect_task(q: str) -> str:
    ql = (q or '').lower().strip()
    if ql.startswith('is there any') or ql.startswith('are there any'):
        return 'exists'
    if any(w in ql for w in ['list all', 'list ', 'show all', 'show ', 'output all', 'output ', 'give all', 'print all', 'name all']):
        return 'list'
    if any(w in ql for w in ['average', 'avg', 'mean']):
        return 'avg'
    if any(w in ql for w in ['maximum', 'max']):
        return 'max'
    if any(w in ql for w in ['minimum', 'min']):
        return 'min'
    if any(w in ql for w in ['how many', 'number of', 'count']):
        return 'count'
    if any(w in ql for w in ['trend', 'over time', 'time-of-day', 'evolve', 'evolves', 'change over time', 'changes over time', 'stable', 'constant', 'increasing', 'decreasing']):
        return 'trend'
    return 'factual'

def is_comparison(q: str) -> bool:
    ql = (q or '').lower()
    return 'compare' in ql or ('sensor' in ql and ' and ' in ql)

In [6]:
# -----------------------------
# Query builders
# -----------------------------
def _cypher_literal(value: Any) -> str:
    """Render a Python value as a Cypher literal (for documentation/gold strings only)."""
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    # Default: JSON-encode string for correct quoting/escaping
    return json.dumps(str(value), ensure_ascii=False)


def render_cypher(template: str, params: Dict[str, Any]) -> str:
    """Replace $param placeholders with literal values.

    The pipeline runs Cypher as a plain string (no parameters). Gold labels therefore
    store a rendered string with embedded literals, while execution can still use
    parameterized Cypher for safety.
    """
    rendered = template
    for key, val in params.items():
        rendered = re.sub(rf"\${re.escape(key)}\b", _cypher_literal(val), rendered)
    return rendered


def _match_measurements(sensor_id: Optional[int], epoch_id: Optional[int]) -> Tuple[str, Dict[str, Any]]:
    parts = []
    params: Dict[str, Any] = {}
    if sensor_id is not None:
        parts.append("MATCH (s:Sensor {mote_id: $mote_id})-[:GENERATED]->(m:Measurement)")
        params["mote_id"] = sensor_id
    else:
        parts.append("MATCH (m:Measurement)")

    if epoch_id is not None:
        parts.append("MATCH (m)-[:EPOCH_IN]->(e:Epoch {epoch_id: $epoch_id})")
        params["epoch_id"] = epoch_id
    return "\n".join(parts), params


def fetch_records(sensor_id: Optional[int], epoch_id: Optional[int], limit: int) -> Tuple[int, List[Dict[str, Any]], str]:
    match, params = _match_measurements(sensor_id, epoch_id)

    count_cypher = f"{match}\nRETURN count(m) AS total"
    total = int(run_cypher(count_cypher, params)[0]["total"])

    if sensor_id is None:
        # add sensor id for global epoch lists
        match2 = match + "\nOPTIONAL MATCH (s:Sensor)-[:GENERATED]->(m)"
    else:
        match2 = match + "\nWITH m, s"

    data_cypher = (
        f"{match2}\n"
        "RETURN "
        "coalesce(s.mote_id, m.moteid, m.mote_id) AS mote_id, "
        "m.time AS time, "
        "m.temperature AS temperature, "
        "m.humidity AS humidity, "
        "m.light AS light, "
        "m.voltage AS voltage "
        "ORDER BY time, mote_id "
        "LIMIT $limit"
    )
    data_cypher = "".join(data_cypher)
    exec_params = {**params, "limit": int(limit)}
    rows = run_cypher(data_cypher, exec_params)
    return total, rows, render_cypher(data_cypher, exec_params)


def fetch_single_value(sensor_id: int, ts_prefix: str, metric: str) -> Tuple[Optional[float], str]:
    cypher = (
        "MATCH (s:Sensor {mote_id: $mote_id})-[:GENERATED]->(m:Measurement)\n"
        "WHERE m.time STARTS WITH $ts_prefix\n"
        f"RETURN m.time AS time, m.{metric} AS value\n"
        "ORDER BY m.time ASC\n"
        "LIMIT 1"
    )
    params = {"mote_id": sensor_id, "ts_prefix": ts_prefix}
    rows = run_cypher(cypher, params)
    if not rows:
        return None, render_cypher(cypher, params)
    v = rows[0].get("value")
    return (float(v) if v is not None else None), render_cypher(cypher, params)


def fetch_aggregate(sensor_id: Optional[int], epoch_id: Optional[int], metric: Optional[str], agg: str) -> Tuple[Optional[float], str]:
    match, params = _match_measurements(sensor_id, epoch_id)

    if agg == "count":
        cypher = f"{match}\nRETURN count(m) AS value"
        rows = run_cypher(cypher, params)
        v = rows[0].get("value") if rows else None
        return (float(v) if v is not None else None), render_cypher(cypher, params)

    if not metric:
        return None, ""

    fn = {"avg": "avg", "min": "min", "max": "max"}[agg]
    cypher = f"{match}\nRETURN {fn}(m.{metric}) AS value"
    rows = run_cypher(cypher, params)
    v = rows[0].get("value") if rows else None
    return (float(v) if v is not None else None), render_cypher(cypher, params)


def fetch_exists(sensor_id: int, metric: str) -> Tuple[int, str]:
    cypher = (
        "MATCH (s:Sensor {mote_id: $mote_id})-[:GENERATED]->(m:Measurement)\n"
        f"WHERE m.{metric} IS NOT NULL\n"
        "RETURN count(m) AS value"
    )
    params = {"mote_id": sensor_id}
    rows = run_cypher(cypher, params)
    v = rows[0].get('value') if rows else None
    count = int(v) if v is not None else 0
    return count, render_cypher(cypher, params)


def fetch_series_for_trend(sensor_id: int, metric: str) -> Tuple[List[Tuple[datetime, float]], str]:
    cypher = (
        "MATCH (s:Sensor {mote_id: $mote_id})-[:GENERATED]->(m:Measurement)\n"
        f"WHERE m.{metric} IS NOT NULL\n"
        f"RETURN m.time AS time, m.{metric} AS value\n"
        "ORDER BY m.time ASC"
    )
    params = {"mote_id": sensor_id}
    rows = run_cypher(cypher, params)
    series: List[Tuple[datetime, float]] = []
    for r in rows:
        t = r.get("time")
        v = r.get("value")
        if t is None or v is None:
            continue
        try:
            dt = datetime.fromisoformat(str(t))
        except Exception:
            try:
                dt = datetime.fromisoformat(str(t).split(".")[0])
            except Exception:
                continue
        try:
            fv = float(v)
        except Exception:
            continue
        series.append((dt, fv))
    return series, render_cypher(cypher, params)

In [7]:
# -----------------------------
# Rendering helpers (English, plain text)
# -----------------------------
def render_list(total: int, rows: List[Dict[str, Any]], max_lines: int) -> str:
    if total <= 0:
        return 'No data available.'

    lines: List[str] = []
    for r in rows[:max_lines]:
        kv = []
        order = ['mote_id', 'time', 'temperature', 'humidity', 'light', 'voltage']
        for k in order:
            if k in r and r[k] is not None:
                kv.append(f"{k}={r[k]}")
        for k, v in r.items():
            if k in order:
                continue
            if v is None:
                continue
            kv.append(f"{k}={v}")
        lines.append(' '.join(kv))

    if total > max_lines:
        lines.append(f"... (additional {total - max_lines} measurements omitted)")
    return '\n'.join(lines)

def _metric_label(metric: str) -> str:
    return {
        'temperature': 'Temperature',
        'humidity': 'Humidity',
        'light': 'Light',
        'voltage': 'Voltage',
    }.get(metric, metric)

def _unit(metric: str) -> str:
    return {
        'temperature': '°C',
        'humidity': '%',
        'light': 'Lux',
        'voltage': 'V',
    }.get(metric, '')

def render_factual(sensor_id: int, ts_prefix: str, metric: str, value: Optional[float]) -> str:
    if value is None:
        return 'No data available.'
    label = _metric_label(metric)
    unit = _unit(metric)
    return f"{label} of sensor {sensor_id}.0 at {ts_prefix.replace('T', ' ')}: {value}{unit}"

def render_aggregate(agg: str, metric: Optional[str], value: Optional[float], epoch_id: Optional[int], sensor_id: Optional[int]) -> str:
    if value is None:
        return 'No data available.'

    scope = []
    if sensor_id is not None:
        scope.append(f"Sensor {sensor_id}.0")
    if epoch_id is not None:
        scope.append(f"Epoch {epoch_id}")
    scope_str = ' in '.join(scope) if scope else 'all data'

    if agg == 'count':
        if int(value) == 0:
            return 'No data available.'
        return f"Number of measurements ({scope_str}): {int(value)}"

    assert metric is not None
    label = _metric_label(metric)
    unit = _unit(metric)
    prefix = {'avg': 'Average', 'min': 'Minimum', 'max': 'Maximum'}[agg]
    return f"{prefix} {label} ({scope_str}): {value}{unit}"

def render_exists(sensor_id: int, metric: str, count: int) -> str:
    # For evaluation we treat 'no rows' as no-data; otherwise a minimal affirmative.
    if count <= 0:
        return 'No data available.'
    label = _metric_label(metric)
    return f"{label} data is available for sensor {sensor_id}.0."

def trend_direction(series: List[Tuple[datetime, float]]) -> str:
    if len(series) < 2:
        return 'undetermined'
    t0 = series[0][0]
    x = np.array([(t - t0).total_seconds() for t, _ in series], dtype=float)
    y = np.array([v for _, v in series], dtype=float)
    duration = float(x[-1] - x[0])
    if duration <= 0:
        return 'undetermined'
    slope, _ = np.polyfit(x, y, 1)
    delta = slope * duration
    std = float(np.std(y))
    if std == 0.0 or abs(delta) < 0.1 * std:
        return 'constant'
    return 'increasing' if slope > 0 else 'decreasing'

def render_trend(sensor_id: int, metric: str, direction: str) -> str:
    label = _metric_label(metric)
    return f"The trend of {label} for sensor {sensor_id}.0 is: {direction}."

def render_trend_compare(sensor_a: int, sensor_b: int, metric: str, dir_a: str, dir_b: str) -> str:
    label = _metric_label(metric)
    return f"Trend comparison for {label}: sensor {sensor_a}.0={dir_a}, sensor {sensor_b}.0={dir_b}."

In [8]:
# -----------------------------
# Goldlabel generation
# -----------------------------
@dataclass
class GoldResult:
    gold_answer: str
    cypher: Optional[str] = None
    meta: Optional[Dict[str, Any]] = None

def generate_gold(question: str) -> GoldResult:
    task = detect_task(question)
    epoch_id = parse_epoch(question)
    sensor_id = parse_sensor(question)
    metric = parse_metric(question)
    ts_prefix = parse_datetime_prefix(question)

    # Trend comparison: try to extract two sensors
    if task == 'trend' and is_comparison(question):
        sensors = [int(float(x)) for x in re.findall(r"sensor\s*(\d+(?:\.0)?)", question, flags=re.IGNORECASE)]
        sensors = [s for s in sensors if isinstance(s, int)]
        sensors = list(dict.fromkeys(sensors))  # unique preserve order
        if len(sensors) >= 2 and metric:
            s1, s2 = sensors[0], sensors[1]
            series1, cy1 = fetch_series_for_trend(s1, metric)
            series2, cy2 = fetch_series_for_trend(s2, metric)
            d1 = trend_direction(series1)
            d2 = trend_direction(series2)
            if d1 == 'undetermined' or d2 == 'undetermined':
                return GoldResult('No data available.', cypher=f"{cy1}\n---\n{cy2}")
            return GoldResult(render_trend_compare(s1, s2, metric, d1, d2), cypher=f"{cy1}\n---\n{cy2}")

    if task == 'exists':
        if sensor_id is None or metric is None:
            return GoldResult('No data available.')
        count, cypher = fetch_exists(sensor_id, metric)
        return GoldResult(render_exists(sensor_id, metric, count), cypher=cypher, meta={'count': count})

    if task == 'list':
        total, rows, cypher = fetch_records(sensor_id=sensor_id, epoch_id=epoch_id, limit=LIST_MAX_LINES)
        return GoldResult(render_list(total, rows, LIST_MAX_LINES), cypher=cypher, meta={'total': total})

    if task in {'avg', 'min', 'max', 'count'}:
        value, cypher = fetch_aggregate(sensor_id=sensor_id, epoch_id=epoch_id, metric=metric, agg=task)
        return GoldResult(render_aggregate(task, metric, value, epoch_id, sensor_id), cypher=cypher)

    if task == 'trend':
        if sensor_id is None or metric is None:
            return GoldResult('No data available.')
        series, cypher = fetch_series_for_trend(sensor_id, metric)
        direction = trend_direction(series)
        if direction == 'undetermined':
            return GoldResult('No data available.', cypher=cypher)
        return GoldResult(render_trend(sensor_id, metric, direction), cypher=cypher)

    # factual
    if sensor_id is None or metric is None or ts_prefix is None:
        return GoldResult('No data available.')
    value, cypher = fetch_single_value(sensor_id, ts_prefix, metric)
    return GoldResult(render_factual(sensor_id, ts_prefix, metric, value), cypher=cypher)


# Preview on first 5 questions
for row in dataset[:5]:
    q = row.get('question') or row.get('query') or ''
    gr = generate_gold(q)
    print('Q:', q)
    print('G:', gr.gold_answer)
    print('---')

Q: What temperature did sensor 19.0 measure on 2004-02-28 at 00:58:46?
G: Temperature of sensor 19.0 at 2004-02-28 00:58:46: 19.7336°C
---
Q: What temperature did sensor 25.0 measure on 2004-02-28 at 00:58:46?
G: Temperature of sensor 25.0 at 2004-02-28 00:58:46: 19.616°C
---
Q: What temperature did sensor 24.0 measure on 2004-02-28 at 00:58:46?
G: Temperature of sensor 24.0 at 2004-02-28 00:58:46: 19.7728°C
---
Q: What light did sensor 3.0 measure on 2004-02-28 at 00:58:46?
G: Light of sensor 3.0 at 2004-02-28 00:58:46: 50.6Lux
---
Q: What temperature did sensor 6.0 measure on 2004-02-28 at 00:58:46?
G: Temperature of sensor 6.0 at 2004-02-28 00:58:46: 20.4196°C
---


In [9]:
# -----------------------------
# Run generation for full dataset and write JSONL
# -----------------------------
out_rows: List[Dict[str, Any]] = []

for row in tqdm(dataset, desc='generating goldlabels'):
    q = row.get('question') or row.get('query') or ''
    gr = generate_gold(q)
    new_row = dict(row)
    new_row['gold_answer'] = gr.gold_answer
    # Optional debug fields (keep or delete as needed)
    new_row['gold_cypher'] = gr.cypher
    if gr.meta:
        new_row['gold_meta'] = gr.meta
    out_rows.append(new_row)

save_jsonl(OUTPUT_PATH, out_rows)
print('Wrote:', OUTPUT_PATH, 'rows=', len(out_rows))

generating goldlabels: 100%|██████████| 100/100 [00:00<00:00, 215.75it/s]

Wrote: /app/tests/datasets/eval_dataset_with_gold.jsonl rows= 100


In [10]:
# Cleanup (optional)
# driver.close()